# Notebook 17 — Budgeted structured mask selection and physical channel surgery

**Purpose.** Turn score rankings into physically smaller CNN1D models and run
the Stage-B screening comparison at matched approximate removable-FLOP budgets.

This notebook uses **validation** results for method elimination. It does not
create the final test-set claims; confirmatory multi-seed evaluation belongs in
Notebook 19 after SABER passes the early gates.

**Methods in the starter screen**

- random
- magnitude
- Taylor
- Fisher
- Robust Semantic Boundary Leverage (R-SBL)

**Budgets:** 25%, 40%, and 55% of the directly removable channel cost by
default. The realised architecture, parameter count, and forward validity are
recorded for every point.

In [ ]:
# Colab/repository bootstrap
from pathlib import Path
import os, sys, json, subprocess, platform

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

# Override with %env SABER_REPO=/your/path if your repository is elsewhere.
candidates = [
    os.environ.get("SABER_REPO"),
    "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression",
    str(Path.cwd()),
]
REPO = None
for candidate in candidates:
    if not candidate:
        continue
    p = Path(candidate).expanduser()
    if (p / "src").is_dir() and (p / "config").is_dir():
        REPO = p.resolve()
        break
if REPO is None:
    raise FileNotFoundError(
        "Repository not found. Set SABER_REPO or edit the candidate path."
    )
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print("Repository:", REPO)
print("Python:", sys.version.split()[0], "| Platform:", platform.platform())

In [ ]:
# Install only the small SABER extension requirements.
# The original repository requirements must already be installed.
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-saber.txt"],
        check=True,
    )

In [ ]:
import yaml
from src.saber.adapters import SaberRepo

repo = SaberRepo.discover(REPO)
with open(REPO / "config" / "saber.yaml", "r", encoding="utf-8") as handle:
    SABER_CFG = yaml.safe_load(handle)

OUTPUT_ROOT = repo.output_root
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("SABER output root:", OUTPUT_ROOT)
print("Config branch:", SABER_CFG["project"]["branch"])

In [ ]:
# Repository bridge: auto-discovery first, explicit override second.
#
# If auto-discovery fails, set these objects using the same loader/model
# construction cells from the completed Computer Networks notebooks:
#   TRAIN_LOADER = ...
#   VAL_LOADER = ...
#   TEST_LOADER = ...
#   MODEL = ...
#   CLASS_NAMES = [...]
#
# MODEL must be the uncompressed CNN1D anchor and loaders must use the frozen
# train/validation/test split.

import torch
from src.saber.adapters import (
    auto_discover_loaders,
    auto_build_cnn,
    discover_anchor_checkpoint,
    infer_class_names_from_results,
    unpack_batch,
)

TRAIN_LOADER = globals().get("TRAIN_LOADER")
VAL_LOADER = globals().get("VAL_LOADER")
TEST_LOADER = globals().get("TEST_LOADER")
MODEL = globals().get("MODEL")
CLASS_NAMES = globals().get("CLASS_NAMES")

if any(obj is None for obj in (TRAIN_LOADER, VAL_LOADER, TEST_LOADER)):
    TRAIN_LOADER, VAL_LOADER, TEST_LOADER, _DATA_BUNDLE = auto_discover_loaders(repo)

first_batch = next(iter(VAL_LOADER))
x0, y0, env0 = unpack_batch(first_batch)
raw_example = x0[: min(8, len(x0))].float()

if CLASS_NAMES is None:
    CLASS_NAMES = infer_class_names_from_results(repo)

if MODEL is None:
    checkpoint = discover_anchor_checkpoint(repo)
    MODEL, MODEL_FACTORY_ERRORS = auto_build_cnn(
        n_features=int(x0.shape[-1]),
        n_classes=len(CLASS_NAMES),
        checkpoint=checkpoint,
    )
    print("Loaded checkpoint:", checkpoint)
    if MODEL_FACTORY_ERRORS:
        print("Model factory attempts that were skipped:", MODEL_FACTORY_ERRORS)

# Infer whether the historical CNN expects [B,F] and unsqueezes internally or
# expects an explicit [B,1,F] tensor. This decision is frozen for the notebook.
MODEL_INPUT_MODE = None
probe_out = None
candidate_inputs = [("raw", raw_example)]
if raw_example.ndim == 2:
    candidate_inputs.append(("unsqueeze_channel", raw_example.unsqueeze(1)))
errors = {}
MODEL.eval()
for mode, candidate in candidate_inputs:
    try:
        with torch.no_grad():
            probe_out = MODEL(candidate)
        MODEL_INPUT_MODE = mode
        EXAMPLE_INPUT = candidate
        break
    except Exception as exc:
        errors[mode] = repr(exc)

if MODEL_INPUT_MODE is None:
    raise RuntimeError(
        "Could not infer the CNN input convention. Set EXAMPLE_INPUT and "
        "MODEL_INPUT manually. Attempts: " + json.dumps(errors, indent=2)
    )

def MODEL_INPUT(x):
    if MODEL_INPUT_MODE == "unsqueeze_channel" and x.ndim == 2:
        return x.unsqueeze(1)
    return x

if probe_out.shape[1] != len(CLASS_NAMES):
    raise RuntimeError(
        f"Model outputs {probe_out.shape[1]} classes but CLASS_NAMES has "
        f"{len(CLASS_NAMES)} entries. Supply the exact label-encoder order."
    )

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL = MODEL.to(DEVICE)
EXAMPLE_INPUT = EXAMPLE_INPUT.to(DEVICE)
print("Device:", DEVICE)
print("Model:", type(MODEL).__name__)
print("Classes:", len(CLASS_NAMES))
print("Input convention:", MODEL_INPUT_MODE, tuple(EXAMPLE_INPUT.shape))

## 1. Load frozen score and graph artefacts

In [ ]:
from copy import deepcopy
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch

from src.saber.taxonomy import ciciot2023_taxonomy, DEFAULT_COST_PROFILES
from src.saber.selectors import (
    select_groups_greedy,
    selection_to_prune_map,
    summarize_layer_widths,
)
from src.saber.surgery import (
    prune_cnn1d_channels,
    count_parameters,
    count_nonzero_parameters,
    profile_forward_flops,
)
from src.saber.adapters import collect_logits, unpack_batch
from src.saber.metrics import (
    action_weighted_boundary_inversion_rate,
    full_model_audit,
)

OUT = OUTPUT_ROOT / "17_structured_selection"
OUT.mkdir(parents=True, exist_ok=True)

taxonomy = ciciot2023_taxonomy(CLASS_NAMES)
scores = pd.read_csv(
    OUTPUT_ROOT / "16_score_validation" / "group_scores_with_r_sbl.csv"
)
robust_graph = pd.read_csv(
    OUTPUT_ROOT / "14_risk_graph" / "asvg_edges_robust.csv"
)

METHOD_SCORE = {
    "random": "random",
    "magnitude": "magnitude",
    "taylor": "taylor",
    "fisher": "fisher",
    "r_sbl": "r_sbl",
}
BUDGETS = [float(v) for v in SABER_CFG["structured_pruning"]["flops_reduction_budgets"]]
print("Methods:", METHOD_SCORE)
print("Budgets:", BUDGETS)

## 2. Cache teacher validation outputs and class weights

All screening methods receive the same square-root inverse-frequency training
weights and the same fine-tuning budget.

In [ ]:
TEACHER_CACHE = OUTPUT_ROOT / "14_risk_graph" / "validation_teacher_outputs.npz"
cached = np.load(TEACHER_CACHE, allow_pickle=True)
VAL_TEACHER_LOGITS = cached["logits"]
VAL_LABELS = cached["labels"].astype(np.int64)
teacher_audit = full_model_audit(
    VAL_TEACHER_LOGITS, VAL_LABELS, taxonomy, DEFAULT_COST_PROFILES
)
M0_FLOPS = profile_forward_flops(
    MODEL, EXAMPLE_INPUT.to(DEVICE)
)["flops_per_item"]
print("Dense M0 FLOPs per item:", M0_FLOPS)

counts = np.zeros(taxonomy.n_classes, dtype=np.int64)
for batch in TRAIN_LOADER:
    _, y, _ = unpack_batch(batch)
    y_np = y.detach().cpu().numpy() if hasattr(y, "detach") else np.asarray(y)
    counts += np.bincount(y_np.astype(int), minlength=taxonomy.n_classes)
weights = np.zeros_like(counts, dtype=np.float64)
present = counts > 0
weights[present] = 1.0 / np.sqrt(counts[present])
weights[present] /= weights[present].mean()
CLASS_WEIGHTS = torch.tensor(weights, dtype=torch.float32, device=DEVICE)
print(pd.DataFrame({"class": CLASS_NAMES, "train_count": counts, "weight": weights}).head())

## 3. Shared evaluation and screening fine-tuning functions

In [ ]:
from sklearn.metrics import f1_score
from torch import nn

@torch.no_grad()
def evaluate_validation(model):
    logits, labels, _ = collect_logits(
        model, VAL_LOADER, device=DEVICE, input_transform=MODEL_INPUT
    )
    audit = full_model_audit(logits, labels, taxonomy, DEFAULT_COST_PROFILES)
    awbir, _ = action_weighted_boundary_inversion_rate(
        VAL_TEACHER_LOGITS,
        logits,
        labels,
        robust_graph,
        weight_column="robust_weight",
    )
    audit["awbir"] = float(awbir)
    return logits, labels, audit

def screening_finetune(model, *, epochs=8, lr=1e-3, patience=3):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)
    best_state = deepcopy(model.state_dict())
    best_f1 = -np.inf
    stale = 0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        seen = 0
        for batch in TRAIN_LOADER:
            x, y, _ = unpack_batch(batch)
            x = MODEL_INPUT(x.to(DEVICE))
            y = y.to(DEVICE).long()
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            running_loss += float(loss.detach().cpu()) * len(y)
            seen += len(y)

        _, _, audit = evaluate_validation(model)
        val_f1 = float(audit["fine_macro_f1"])
        history.append({
            "epoch": epoch,
            "train_loss": running_loss / max(seen, 1),
            "val_macro_f1": val_f1,
            "val_awbir": audit["awbir"],
            "val_hsr_balanced_soc": audit["hsr_balanced_soc"],
        })
        if val_f1 > best_f1 + 1e-6:
            best_f1 = val_f1
            best_state = deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1
        if stale >= patience:
            break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)

## 4. Select, physically prune, and screen each operating point

The loop is resumable at the result-table level. Checkpoints are stored under a
method/budget-specific path and include the selection/audit metadata.

In [ ]:
RESULT_CSV = OUT / "structured_screening_results.csv"
if RESULT_CSV.exists():
    previous = pd.read_csv(RESULT_CSV)
    completed = set(zip(previous["method"], previous["budget"]))
    result_rows = previous.to_dict(orient="records")
else:
    completed = set()
    result_rows = []

for method, score_column in METHOD_SCORE.items():
    for budget in BUDGETS:
        key = (method, budget)
        if key in completed:
            continue
        print(f"\n=== {method} @ {budget:.0%} ===")
        selection, selection_summary = select_groups_greedy(
            scores,
            importance_column=score_column,
            cost_column="flops_cost",
            target_reduction_fraction=budget,
            minimum_remaining_per_layer=int(
                SABER_CFG["groups"]["minimum_remaining_per_layer"]
            ),
        )
        prune_map = selection_to_prune_map(selection)
        student, surgery_audit = prune_cnn1d_channels(
            MODEL,
            prune_map,
            EXAMPLE_INPUT.to(DEVICE),
            minimum_remaining_per_layer=int(
                SABER_CFG["groups"]["minimum_remaining_per_layer"]
            ),
        )
        pre_logits, pre_labels, pre_audit = evaluate_validation(student)
        student, history = screening_finetune(student)
        post_logits, post_labels, post_audit = evaluate_validation(student)

        tag = f"{method}_b{int(round(100*budget)):02d}"
        selection.to_csv(OUT / f"{tag}_selection.csv", index=False)
        summarize_layer_widths(selection).to_csv(
            OUT / f"{tag}_layer_widths.csv", index=False
        )
        surgery_audit.to_csv(OUT / f"{tag}_surgery_audit.csv", index=False)
        history.to_csv(OUT / f"{tag}_finetune_history.csv", index=False)
        torch.save(
            {
                "state_dict": student.cpu().state_dict(),
                "method": method,
                "score_column": score_column,
                "budget": budget,
                "selection_summary": selection_summary,
                "surgery_audit": surgery_audit.to_dict(orient="records"),
                "class_names": CLASS_NAMES,
            },
            OUT / f"{tag}_checkpoint.pt",
        )
        student = student.to(DEVICE)

        row = {
            "method": method,
            "score_column": score_column,
            "budget": budget,
            "target_reduction_fraction": selection_summary["target_reduction_fraction"],
            "achieved_direct_cost_reduction": selection_summary["achieved_reduction_fraction"],
            "selected_groups": selection_summary["selected_groups"],
            "parameters_m0": count_parameters(MODEL),
            "parameters_student": count_parameters(student),
            "parameter_reduction_fraction": 1.0 - count_parameters(student) / count_parameters(MODEL),
            "flops_per_item_m0": M0_FLOPS,
            "flops_per_item_student": profile_forward_flops(
                student, EXAMPLE_INPUT.to(DEVICE)
            )["flops_per_item"],
            "flops_reduction_fraction": 1.0 - profile_forward_flops(
                student, EXAMPLE_INPUT.to(DEVICE)
            )["flops_per_item"] / M0_FLOPS,
            "pre_macro_f1": pre_audit["fine_macro_f1"],
            "post_macro_f1": post_audit["fine_macro_f1"],
            "post_family_macro_f1": post_audit["family_macro_f1"],
            "post_attack_to_benign": post_audit["attack_to_benign_rate"],
            "post_benign_to_attack": post_audit["benign_to_attack_rate"],
            "post_awbir": post_audit["awbir"],
            "post_hsr_balanced_soc": post_audit["hsr_balanced_soc"],
            "post_ece15": post_audit["ece15"],
            "epochs_run": int(len(history)),
        }
        result_rows.append(row)
        pd.DataFrame(result_rows).to_csv(RESULT_CSV, index=False)
        print(row)

results = pd.DataFrame(result_rows).sort_values(["budget", "method"])
display(results)

## 5. Stage-B Pareto and gate diagnostics

This is a screening comparison, not the confirmatory result. R-SBL should show a
consistent semantic advantage at matched physical cost. A method that wins only
macro-F1 but not AWBIR/HSR does not justify the proposed paper.

In [ ]:
import matplotlib.pyplot as plt
from src.saber.selectors import pareto_nondominated

results["pareto_semantic"] = pareto_nondominated(
    results,
    maximize=["post_macro_f1", "post_family_macro_f1"],
    minimize=["post_awbir", "post_hsr_balanced_soc"],
)
results.to_csv(RESULT_CSV, index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for method, frame in results.groupby("method"):
    frame = frame.sort_values("flops_reduction_fraction")
    axes[0].plot(
        frame["flops_reduction_fraction"],
        frame["post_awbir"],
        marker="o",
        label=method,
    )
    axes[1].plot(
        frame["flops_reduction_fraction"],
        frame["post_family_macro_f1"],
        marker="o",
        label=method,
    )
axes[0].set_xlabel("Realised dense FLOP reduction")
axes[0].set_ylabel("Validation AWBIR (lower is better)")
axes[1].set_xlabel("Realised dense FLOP reduction")
axes[1].set_ylabel("Validation family macro-F1")
axes[0].legend()
axes[1].legend()
fig.suptitle("Stage-B structured pruning screen")
fig.tight_layout()
fig.savefig(OUT / "structured_screening_pareto.png", dpi=250)
plt.show()

# Simple provisional gate: R-SBL wins at least two of AWBIR, HSR, family F1
# against the best generic method at the nearest achieved reduction.
gate_details = []
for budget in BUDGETS:
    frame = results[np.isclose(results["budget"], budget)]
    saber = frame[frame["method"] == "r_sbl"]
    generic = frame[frame["method"].isin(["magnitude", "taylor", "fisher"])]
    if saber.empty or generic.empty:
        continue
    s = saber.iloc[0]
    best_awbir = generic["post_awbir"].min()
    best_hsr = generic["post_hsr_balanced_soc"].min()
    best_family = generic["post_family_macro_f1"].max()
    wins = {
        "awbir": bool(s["post_awbir"] < best_awbir),
        "hsr": bool(s["post_hsr_balanced_soc"] < best_hsr),
        "family_f1": bool(s["post_family_macro_f1"] > best_family),
    }
    gate_details.append({"budget": budget, **wins, "wins": sum(wins.values())})

gate_pass = any(row["wins"] >= 2 for row in gate_details)
gate = {"gate": "G2_core_pruning_screen", "passed": gate_pass, "details": gate_details}
(OUT / "G2_screen_gate.json").write_text(json.dumps(gate, indent=2), encoding="utf-8")
gate

## 6. Freeze screening artefacts

Proceed to Notebook 18 only if R-SBL is competitive and at least one operating
point has an acceptable semantic trade-off. If not, revisit the graph or score
before adding a more complex loss.

In [ ]:
import subprocess
from src.saber.adapters import save_run_manifest

try:
    git_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
except Exception:
    git_commit = None

artefacts = [
    RESULT_CSV,
    OUT / "structured_screening_pareto.png",
    OUT / "G2_screen_gate.json",
]
save_run_manifest(
    OUT / "manifest.json",
    notebook="17_saber_budgeted_mask_selection.ipynb",
    config=SABER_CFG,
    artifacts=artefacts,
    git_commit=git_commit,
)
print("Notebook 17 complete. G2 provisional pass:", gate_pass)